In [1]:
import os
os.environ["WANDB_MODE"] = "disabled"

In [2]:
!nvidia-smi

Tue Sep 22 13:07:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%cd /content
!git clone https://github.com/Syed1611/curvature-tuning-research.git
%cd /content/curvature-tuning-research/src/curvature-tuning
!git branch --show-current

/content
Cloning into 'curvature-tuning-research'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 175 (delta 42), reused 159 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 1.12 MiB | 13.65 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct


In [4]:
%pip install -q -r requirements.txt

%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━

The reason behind downgrading the numpy version is because of the datasets and loguru library not being compitble with the numpy version preinstalled in the oldest environment in colab. It'll ask to restart just restart and then run the code below this.
#**`DO NOT RUN THE CODE ABOVE AFTER YOU CLICK ON RESTART`**

In [1]:
import torch
import datasets
import numpy
import pandas
import sklearn
import tqdm
import loguru

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("datasets:", datasets.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("tqdm:", tqdm.__version__)
print("loguru:", loguru.__version__)

Torch: 2.6.0+cu124
CUDA: True
GPU: Tesla T4
datasets: 3.4.1
NumPy: 1.26.3
Pandas: 2.2.3
sklearn: 1.5.2
tqdm: 4.66.5
loguru: 0.7.2


In [2]:
from datasets import load_dataset

data_files = {
    "train": "hf://datasets/AI-Lab-Makerere/beans/data/train-00000-of-00001.parquet",
    "validation": "hf://datasets/AI-Lab-Makerere/beans/data/validation-00000-of-00001.parquet",
    "test": "hf://datasets/AI-Lab-Makerere/beans/data/test-00000-of-00001.parquet",
}

beans = load_dataset("parquet", data_files=data_files)

print(beans)
print("Train:", len(beans["train"]))
print("Validation:", len(beans["validation"]))
print("Test:", len(beans["test"]))
print("Columns:", beans["train"].column_names)
print(beans["train"].features)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


data/train-00000-of-00001.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/18.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 1034
    })
    validation: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 133
    })
    test: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 128
    })
})
Train: 1034
Validation: 133
Test: 128
Columns: ['image_file_path', 'image', 'labels']
{'image_file_path': Value(dtype='string', id=None), 'image': Image(mode=None, decode=True, id=None), 'labels': ClassLabel(names=['angular_leaf_spot', 'bean_rust', 'healthy'], id=None)}


In [8]:
import sys

from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_beans",
    train_batch_size=32,
    test_batch_size=800,
    seed=42
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

ModuleNotFoundError: No module named 'utils'

In [ ]:
!python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800

2026-09-15 08:52:37.255 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet18_seed42.log
2026-09-15 08:52:37.257 | INFO     | __main__:main:58 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
2026-09-15 08:52:38.354 | INFO     | __main__:main:85 - Testing baseline...
2026-09-15 08:52:38.369 | INFO     | __main__:main:88 - Number of trainable parameters: 1539
2026-09-15 08:52:38.369 | INFO     | __main__:main:89 - Starting transfer learning...
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: 

In [ ]:
!cat results/base_imagenet_to_beans_resnet18_seed42.json
!cat results/ct_imagenet_to_beans_resnet18_seed42.json

{
  "num_params": 1539,
  "accuracy": 89.0625
}{
  "num_params": 1539,
  "accuracy": 91.40625,
  "beta": 0.78,
  "coeff": 0.5,
  "best_val_acc": 97.74436090225564,
  "val_acc_list": [
    90.97744360902256,
    90.22556390977444,
    94.73684210526316,
    93.98496240601504,
    96.2406015037594,
    95.48872180451127,
    95.48872180451127,
    96.2406015037594,
    97.74436090225564,
    96.99248120300751,
    96.2406015037594,
    96.99248120300751,
    95.48872180451127,
    95.48872180451127,
    94.73684210526316,
    93.23308270676692,
    93.23308270676692,
    94.73684210526316,
    93.23308270676692,
    92.4812030075188,
    92.4812030075188,
    92.4812030075188,
    94.73684210526316,
    92.4812030075188,
    93.98496240601504,
    93.98496240601504,
    89.47368421052632,
    91.72932330827068,
    93.98496240601504,
    91.72932330827068,
    33.08270676691729
  ]
}

In [ ]:
!python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800

2026-09-15 09:06:00.893 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet18_seed43.log
2026-09-15 09:06:00.897 | INFO     | __main__:main:58 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
2026-09-15 09:06:02.067 | INFO     | __main__:main:85 - Testing baseline...
2026-09-15 09:06:02.082 | INFO     | __main__:main:88 - Number of trainable parameters: 1539
2026-09-15 09:06:02.083 | INFO     | __main__:main:89 - Starting transfer learning...
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: 

In [ ]:
!python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800

2026-09-15 09:16:25.708 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet18_seed44.log
2026-09-15 09:16:25.712 | INFO     | __main__:main:58 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
2026-09-15 09:16:26.794 | INFO     | __main__:main:85 - Testing baseline...
2026-09-15 09:16:26.807 | INFO     | __main__:main:88 - Number of trainable parameters: 1539
2026-09-15 09:16:26.807 | INFO     | __main__:main:89 - Starting transfer learning...
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: 

In [ ]:
import json
import numpy as np

seeds = [42, 43, 44]

for method in ["base", "ct"]:
    accuracies = []
    betas = []

    for seed in seeds:
        path = f"results/{method}_imagenet_to_beans_resnet18_seed{seed}.json"

        with open(path) as f:
            result = json.load(f)

        accuracies.append(result["accuracy"])

        if "beta" in result:
            betas.append(result["beta"])

    print(f"\n{method.upper()}")
    print("Individual accuracies:", accuracies)
    print(f"Mean accuracy: {np.mean(accuracies):.2f}%")
    print(f"Std: {np.std(accuracies):.2f}")

    if betas:
        print("Selected betas:", betas)
        print(f"Mean beta: {np.mean(betas):.2f}")


BASE
Individual accuracies: [89.0625, 91.40625, 87.5]
Mean accuracy: 89.32%
Std: 1.61

CT
Individual accuracies: [91.40625, 90.625, 91.40625]
Mean accuracy: 91.15%
Std: 0.37
Selected betas: [0.78, 0.77, 0.79]
Mean beta: 0.78


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/curvature_tuning_research")

for folder in ["results", "logs", "notes", "code_snapshots"]:
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("Research folder:", DRIVE_ROOT)

Research folder: /content/drive/MyDrive/curvature_tuning_research


In [ ]:
import shutil
from pathlib import Path

PROJECT = Path("/content/curvature-tuning")

# Save results
if (PROJECT / "results").exists():
    shutil.copytree(
        PROJECT / "results",
        DRIVE_ROOT / "results",
        dirs_exist_ok=True
    )

# Save logs
if (PROJECT / "logs").exists():
    shutil.copytree(
        PROJECT / "logs",
        DRIVE_ROOT / "logs",
        dirs_exist_ok=True
    )

print("Results and logs saved to Google Drive.")

Results and logs saved to Google Drive.


In [ ]:
summary = """
Curvature Tuning - Original Paper Reproduction
==============================================

Model: ResNet-18
Pretraining: ImageNet
Dataset: Beans

Seeds: 42, 43, 44

BASELINE
Seed 42: 89.0625%
Seed 43: 91.40625%
Seed 44: 87.5000%
Mean: 89.32%
Std: 1.61

S-CT
Seed 42: 91.40625%
Seed 43: 90.6250%
Seed 44: 91.40625%
Mean: 91.15%
Std: 0.37

Selected betas:
Seed 42: 0.78
Seed 43: 0.77
Seed 44: 0.79
Mean beta: 0.78

Average S-CT improvement:
+1.83 percentage points

Environment:
Python: 3.11
PyTorch: 2.6.0+cu124
GPU: Tesla T4
NumPy: 1.26.3
datasets: 3.4.1

Dataset compatibility note:
The authors' original Beans Google Storage URL returned HTTP 403.
The same public Beans train/validation/test splits were loaded through
the AI-Lab-Makerere Hugging Face parquet files.

Dataset splits:
Train: 1034
Validation: 133
Test: 128
"""

summary_path = DRIVE_ROOT / "notes" / "reproduction_summary.txt"
summary_path.write_text(summary)

print("Saved:", summary_path)

Saved: /content/drive/MyDrive/curvature_tuning_research/notes/reproduction_summary.txt


In [ ]:
import subprocess

diff = subprocess.check_output(
    ["git", "diff"],
    cwd="/content/curvature-tuning",
    text=True
)

(DRIVE_ROOT / "code_snapshots" / "baseline_dataset_fix.diff").write_text(diff)

print("Git diff saved.")

Git diff saved.


In [ ]:
!python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --init_beta 0.8